# Proyecto Final Machine Learning
## Fase 2: Modelado y Benchmarking

**Dataset:** UCI HAR — Human Activity Recognition Using Smartphones  
**Equipo:** Grupo XX  
**Integrantes:** [Nombre 1] · [Nombre 2] · [Nombre 3]  
**Fecha:** [Fecha de entrega]

---

> **Objetivo:** Entrenar múltiples modelos de clasificación, evaluarlos con métricas adecuadas y construir una tabla de benchmarking que permita comparar su desempeño de forma rigurosa.

## Configuración del entorno

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score
from sklearn.metrics import (
    accuracy_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print('Librerías cargadas correctamente.')

## 1. Carga de datos desde Fase 1

Reutilizamos los datos preparados en la Fase 1. Ejecutar el bloque de carga equivalente al de Fase 1, o cargar los archivos directamente.

In [ ]:
# Carga de datos (mismo codigo que Fase 1)
import os, urllib.request, zipfile

DATA_PATH = 'UCI HAR Dataset'
ZIP_PATH = 'har_dataset.zip'
INNER_ZIP_PATH = 'UCI HAR Dataset.zip'

if not os.path.exists(DATA_PATH):
    URL = 'https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip'
    print('Descargando dataset...')
    urllib.request.urlretrieve(URL, ZIP_PATH)

    print('Primer unzip...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall('.')
    os.remove(ZIP_PATH)

    # El ZIP descargado contiene otro ZIP adentro
    if os.path.exists(INNER_ZIP_PATH):
        print('Segundo unzip...')
        with zipfile.ZipFile(INNER_ZIP_PATH, 'r') as z:
            z.extractall('.')
        os.remove(INNER_ZIP_PATH)

    print('Dataset listo.')
else:
    print('Dataset ya descargado.')

# Cargar nombres de features y desduplicar
features = pd.read_csv(f'{DATA_PATH}/features.txt', sep=r'\s+', header=None, names=['idx', 'feature'])
feature_names = features['feature'].tolist()

seen = {}
unique_names = []
for name in feature_names:
    if name in seen:
        seen[name] += 1
        unique_names.append(f'{name}_{seen[name]}')
    else:
        seen[name] = 0
        unique_names.append(name)
feature_names = unique_names

X_train = pd.read_csv(f'{DATA_PATH}/train/X_train.txt', sep=r'\s+', header=None, names=feature_names)
y_train = pd.read_csv(f'{DATA_PATH}/train/y_train.txt', sep=r'\s+', header=None, names=['Activity']).squeeze()
X_test  = pd.read_csv(f'{DATA_PATH}/test/X_test.txt',   sep=r'\s+', header=None, names=feature_names)
y_test  = pd.read_csv(f'{DATA_PATH}/test/y_test.txt',   sep=r'\s+', header=None, names=['Activity']).squeeze()

ACTIVITY_LABELS = {1:'WALKING',2:'WALKING_UPSTAIRS',3:'WALKING_DOWNSTAIRS',4:'SITTING',5:'STANDING',6:'LAYING'}
y_train_labels = y_train.map(ACTIVITY_LABELS)
y_test_labels  = y_test.map(ACTIVITY_LABELS)

print(f'Datos cargados: X_train={X_train.shape} | X_test={X_test.shape}')

---
## 2. Métricas de Evaluación

Dado que el problema tiene 6 clases, la accuracy sola no es suficiente. Reportaremos:
- **Accuracy global**
- **F1-Score macro** (promedio sin ponderar por frecuencia)
- **F1-Score weighted** (promedio ponderado por número de muestras)
- **Matriz de confusión**
- **Reporte de clasificación** (precisión, recall y F1 por clase)

> ⚠️ Siempre reportar métricas sobre el **conjunto de prueba** (X_test, y_test). Nunca tomar decisiones finales sobre el set de entrenamiento.

In [ ]:
def evaluar_modelo(nombre, modelo, X_tr, y_tr, X_te, y_te):
    """Entrena el modelo, mide tiempo y calcula métricas sobre test."""
    t0 = time.time()
    modelo.fit(X_tr, y_tr)
    t_fit = time.time() - t0

    y_pred = modelo.predict(X_te)

    acc      = accuracy_score(y_te, y_pred)
    f1_macro = f1_score(y_te, y_pred, average='macro')
    f1_wgt   = f1_score(y_te, y_pred, average='weighted')

    print(f'\n=== {nombre} ===')
    print(f'  Accuracy:     {acc:.4f}')
    print(f'  F1-Macro:     {f1_macro:.4f}')
    print(f'  F1-Weighted:  {f1_wgt:.4f}')
    print(f'  Tiempo fit:   {t_fit:.2f}s')
    print(classification_report(y_te, y_pred))

    return {
        'Modelo': nombre,
        'Accuracy_Test': acc,
        'F1_Macro': f1_macro,
        'F1_Weighted': f1_wgt,
        'Tiempo_Entrenamiento': round(t_fit, 3),
        'modelo_obj': modelo,
        'y_pred': y_pred
    }

resultados = []

---
## 3. Modelo Baseline

Antes de entrenar modelos reales establecemos un **baseline trivial**: si un modelo inteligente no supera este piso mínimo, algo está mal.

In [ ]:
# TODO 1: Entrenar DummyClassifier con estrategia 'most_frequent'
dummy = DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE)
res_dummy = evaluar_modelo('DummyClassifier (most_frequent)', dummy, X_train, y_train_labels, X_test, y_test_labels)
resultados.append(res_dummy)

In [ ]:
# TODO 2: Calcular manualmente la accuracy del clasificador trivial
# (predice siempre la clase más frecuente)
clase_frecuente = y_train_labels.value_counts().idxmax()
porcentaje = y_train_labels.value_counts(normalize=True).max()
print(f'Clase más frecuente: {clase_frecuente} ({porcentaje:.2%} del train)')
print(f'Accuracy baseline esperada en test: ~{(y_test_labels == clase_frecuente).mean():.4f}')

**Análisis:** *[¿Qué accuracy base esperarían superar? ¿Por qué es importante este paso antes de entrenar modelos más complejos?]*

---
## 4. Modelos Clásicos

Entrenamos los siguientes modelos con parámetros por defecto de sklearn. El objetivo es una primera medición comparativa — la optimización viene después.

| # | Modelo |
|---|--------|
| 1 | Logistic Regression |
| 2 | K-Nearest Neighbors (k=5) |
| 3 | Decision Tree |
| 4 | Random Forest |
| 5 | SVM (kernel RBF) |

### Modelo 1: Logistic Regression

In [ ]:
# TODO 3: Entrenar Logistic Regression
lr = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)
res_lr = evaluar_modelo('Logistic Regression', lr, X_train, y_train_labels, X_test, y_test_labels)
resultados.append(res_lr)

In [ ]:
# Matriz de confusión — Logistic Regression
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(y_test_labels, res_lr['y_pred'],
                                      labels=list(ACTIVITY_LABELS.values())),
    display_labels=list(ACTIVITY_LABELS.values())
)
disp.plot(ax=ax, colorbar=False, xticks_rotation=45)
ax.set_title('Matriz de Confusión — Logistic Regression')
plt.tight_layout()
plt.show()

### Modelo 2: K-Nearest Neighbors (k=5)

In [ ]:
# TODO 4: Entrenar KNN
knn = KNeighborsClassifier(n_neighbors=5)
res_knn = evaluar_modelo('KNN (k=5)', knn, X_train, y_train_labels, X_test, y_test_labels)
resultados.append(res_knn)

In [ ]:
# Matriz de confusión — KNN
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(y_test_labels, res_knn['y_pred'],
                                      labels=list(ACTIVITY_LABELS.values())),
    display_labels=list(ACTIVITY_LABELS.values())
)
disp.plot(ax=ax, colorbar=False, xticks_rotation=45)
ax.set_title('Matriz de Confusión — KNN (k=5)')
plt.tight_layout()
plt.show()

### Modelo 3: Decision Tree

In [ ]:
# TODO 5: Entrenar Decision Tree
dt = DecisionTreeClassifier(random_state=RANDOM_STATE)
res_dt = evaluar_modelo('Decision Tree', dt, X_train, y_train_labels, X_test, y_test_labels)
resultados.append(res_dt)

In [ ]:
# Matriz de confusión — Decision Tree
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(y_test_labels, res_dt['y_pred'],
                                      labels=list(ACTIVITY_LABELS.values())),
    display_labels=list(ACTIVITY_LABELS.values())
)
disp.plot(ax=ax, colorbar=False, xticks_rotation=45)
ax.set_title('Matriz de Confusión — Decision Tree')
plt.tight_layout()
plt.show()

### Modelo 4: Random Forest

In [ ]:
# TODO 6: Entrenar Random Forest
rf = RandomForestClassifier(random_state=RANDOM_STATE)
res_rf = evaluar_modelo('Random Forest', rf, X_train, y_train_labels, X_test, y_test_labels)
resultados.append(res_rf)

In [ ]:
# Matriz de confusión — Random Forest
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(y_test_labels, res_rf['y_pred'],
                                      labels=list(ACTIVITY_LABELS.values())),
    display_labels=list(ACTIVITY_LABELS.values())
)
disp.plot(ax=ax, colorbar=False, xticks_rotation=45)
ax.set_title('Matriz de Confusión — Random Forest')
plt.tight_layout()
plt.show()

### Modelo 5: Support Vector Machine (kernel RBF)

In [ ]:
# TODO 7: Entrenar SVM con kernel RBF
# Nota: SVM puede tardar varios minutos en este dataset
svm = SVC(kernel='rbf', random_state=RANDOM_STATE)
res_svm = evaluar_modelo('SVM (RBF)', svm, X_train, y_train_labels, X_test, y_test_labels)
resultados.append(res_svm)

In [ ]:
# Matriz de confusión — SVM
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(y_test_labels, res_svm['y_pred'],
                                      labels=list(ACTIVITY_LABELS.values())),
    display_labels=list(ACTIVITY_LABELS.values())
)
disp.plot(ax=ax, colorbar=False, xticks_rotation=45)
ax.set_title('Matriz de Confusión — SVM (RBF)')
plt.tight_layout()
plt.show()

---
## 5. Tabla de Benchmarking

Comparamos todos los modelos en una tabla única. Esta es la pieza central de la Fase 2.

In [ ]:
# TODO 8: Construir DataFrame de benchmarking
# Excluimos la columna 'modelo_obj' y 'y_pred' que son objetos Python
df_bench = pd.DataFrame([
    {k: v for k, v in r.items() if k not in ('modelo_obj', 'y_pred')}
    for r in resultados
])

# TODO 9: Ordenar por F1_Macro descendente
df_bench = df_bench.sort_values('F1_Macro', ascending=False).reset_index(drop=True)

print(df_bench.to_string(index=False))

In [ ]:
# TODO 10: Graficar barplot comparativo de F1-Macro por modelo
fig, ax = plt.subplots(figsize=(10, 5))
palette = sns.color_palette('viridis', n_colors=len(df_bench))

bars = ax.barh(df_bench['Modelo'], df_bench['F1_Macro'], color=palette)
ax.bar_label(bars, fmt='%.4f', padding=3)
ax.set_xlim(0, 1.05)
ax.set_xlabel('F1-Score Macro')
ax.set_title('Comparación de Modelos — F1-Score Macro (Test Set)', fontsize=13)
plt.tight_layout()
plt.show()

**Análisis:** *[Identifica el mejor modelo y justifica la elección considerando tanto la métrica F1-Macro como el tiempo de entrenamiento. ¿Existe un trade-off entre rendimiento y costo computacional?]*

---
## 6. Validación Cruzada del Mejor Modelo

La evaluación sobre un único split puede ser ruidosa. Usamos cross-validation para obtener una estimación más robusta.

In [ ]:
# TODO 11: Aplicar 5-fold CV sobre el mejor modelo
mejor_nombre = df_bench.iloc[0]['Modelo']
mejor_modelo = next(r['modelo_obj'] for r in resultados if r['Modelo'] == mejor_nombre)

print(f'Aplicando 5-fold CV al modelo: {mejor_nombre}')

cv_scores = cross_val_score(
    mejor_modelo,
    X_train,
    y_train,
    scoring='f1_macro',
    cv=5,
    n_jobs=-1
)

print(f'F1-Macro por fold: {np.round(cv_scores, 4)}')
print(f'Media CV: {cv_scores.mean():.4f} \u00b1 {cv_scores.std():.4f}')

**Análisis CV:** *[¿El desempeño en CV es consistente con el desempeño en test? ¿Hay indicios de sobreajuste (alta varianza entre folds)?]*

---
## 7. Análisis de Actividades Difíciles

No todas las actividades son igual de fáciles de clasificar. Identificamos dónde falla el modelo.

In [ ]:
# TODO 12: Extraer F1 por clase del mejor modelo
y_pred_best = next(r['y_pred'] for r in resultados if r['Modelo'] == mejor_nombre)

report_dict = classification_report(
    y_test_labels, y_pred_best, output_dict=True
)

f1_por_clase = {
    cls: report_dict[cls]['f1-score']
    for cls in list(ACTIVITY_LABELS.values())
    if cls in report_dict
}

df_f1_clase = pd.Series(f1_por_clase, name='F1').sort_values()
print(df_f1_clase)

In [ ]:
# TODO 13: Visualizar F1 por clase
fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#d62728' if v < 0.85 else '#2ca02c' for v in df_f1_clase.values]
df_f1_clase.plot(kind='barh', ax=ax, color=colors)
ax.axvline(0.85, linestyle='--', color='gray', alpha=0.7, label='Umbral 0.85')
ax.set_xlabel('F1-Score')
ax.set_title(f'F1 por Clase — {mejor_nombre}', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

**Análisis de actividades difíciles:**

*[¿Cuáles son las dos actividades con menor F1? Observa la matriz de confusión del mejor modelo y describe con qué otras actividades se confunden. ¿Tiene sentido físico esta confusión? (Pista: considera qué actividades tienen señales de acelerómetro similares.)]*

---
## 8. Resumen de la Fase 2

In [ ]:
print('=' * 55)
print('RESUMEN FASE 2')
print('=' * 55)
print(f'Modelos entrenados:    {len(df_bench)}')
print(f'Mejor modelo:          {df_bench.iloc[0]["Modelo"]}')
print(f'F1-Macro (test):       {df_bench.iloc[0]["F1_Macro"]:.4f}')
print(f'Accuracy (test):       {df_bench.iloc[0]["Accuracy_Test"]:.4f}')
print(f'Media CV F1-Macro:     {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print('=' * 55)
print('\nTabla completa:')
print(df_bench[['Modelo','Accuracy_Test','F1_Macro','F1_Weighted','Tiempo_Entrenamiento']].to_string(index=False))